# T1 ExpansionNet v2 — Flickr30k pseudo-caption pilot

Pilot này là gate bắt buộc trước full run. Ba pilot T1, T2 và T3 độc lập và có
thể chạy song song trong ba phiên Kaggle. Không đọc Validation caption và không
mở Held-out Test.

## Kaggle input bắt buộc — đúng 5 input

1. Output `F30K_Data`.
2. Output `F30K_Base`.
3. Output `F30K_T1_Handoff`.
4. Dataset `Flickr Image dataset`.
5. Kaggle Model `expansionnet_v2_ieee_bigdata2023` (source ngoại tuyến đã khóa).

## Runtime

- Accelerator: GPU T4; một GPU là đủ.
- Internet: Off.
- T1/T2: 128 ảnh; T3: 100 optimizer-step.
- Pilot T3 bị hủy trọng số sau đo thời gian; full run phải khởi tạo lại.


## Cell 1 — Input gate, Flickr30k contract và image index

In [1]:

EXPERIMENT_ID = "F30K_T1_Pseudo_Pilot"
TEACHER_ID = "T1"
HANDOFF_RESULT_NAME = "F30K_T1_Handoff_RESULT.json"

import csv, hashlib, json, math, os, random, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_WORK_ROOT", "/kaggle/working")) / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def image_index(required_names):
    candidates = [p for p in INPUT_ROOT.rglob("*.jpg") if p.name in required_names]
    by_name = {}
    duplicate_aliases = 0
    for path in candidates:
        old = by_name.get(path.name)
        if old is None:
            by_name[path.name] = path
            continue
        if old.resolve() == path.resolve():
            continue
        assert old.stat().st_size == path.stat().st_size, (
            "Ảnh trùng tên nhưng khác nội dung/kích thước", path.name, old, path
        )
        duplicate_aliases += 1
        by_name[path.name] = min(
            (old, path), key=lambda p: (len(p.parts), len(str(p)), str(p))
        )
    missing = sorted(required_names - set(by_name))
    assert not missing, ("Thiếu ảnh Flickr30k", len(missing), missing[:10])
    return by_name, duplicate_aliases

DATA_PATH = unique_named("f30k_train.parquet")
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
BASE_READY_PATH = unique_named("F30K_BASE_READY.json")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")
HANDOFF_PATH = unique_named(HANDOFF_RESULT_NAME)

data_contract = read_json(DATA_CONTRACT_PATH)
base_ready = read_json(BASE_READY_PATH)
vocab = read_json(VOCAB_PATH)
handoff = read_json(HANDOFF_PATH)

assert data_contract["status"] == "complete"
assert data_contract["train_image_count"] == 29000
assert data_contract["validation_image_count"] == 1014
assert data_contract["test_image_count"] == 1000
assert data_contract["vocabulary_size"] == 7456
assert data_contract["test_set_status"] == "sealed"
assert data_contract["test_caption_exported"] is False
assert base_ready["status"] == "authorized"
assert base_ready["test_set_status"] == "sealed"
assert base_ready["test_set_used"] is False
assert handoff["teacher_id"] == TEACHER_ID
assert handoff["flickr30k"]["test_set_status"] == "sealed"
assert handoff["flickr30k"]["test_set_used"] is False
assert sha256(DATA_PATH) == data_contract["artifacts"]["train"]["sha256"]
assert sha256(VOCAB_PATH) == data_contract["artifacts"]["vocabulary"]["sha256"]

train_df = pd.read_parquet(DATA_PATH)
assert train_df["image_id"].nunique() == 29000
assert len(train_df) == 145000
train_ids = sorted(int(x) for x in train_df["image_id"].unique())
required_names = {f"{image_id}.jpg" for image_id in train_ids}
IMAGE_BY_NAME, DUPLICATE_ALIASES = image_index(required_names)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
assert torch.cuda.is_available(), "Notebook pilot yêu cầu GPU T4."
DEVICE = torch.device("cuda:0")

print("Common gate             : PASSED")
print("Experiment              :", EXPERIMENT_ID)
print("Teacher                 :", TEACHER_ID)
print("Flickr30k Train         : 29,000 images")
print("Vocabulary              : 7,456")
print("Nested image aliases    :", DUPLICATE_ALIASES, "(deduplicated)")
print("Held-out Test           : SEALED / NOT READ")


Common gate             : PASSED
Experiment              : F30K_T1_Pseudo_Pilot
Teacher                 : T1
Flickr30k Train         : 29,000 images
Vocabulary              : 7,456
Nested image aliases    : 29000 (deduplicated)
Held-out Test           : SEALED / NOT READ


## Cell 2 — Teacher workload pilot

In [2]:

import gc, pickle, re, unicodedata
from argparse import Namespace
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

assert handoff["status"] == "passed"
assert handoff["method_contract"]["selected_method"] == "Sequence KD"
assert handoff["method_contract"]["direct_logit_kd_on_f30k"] is False
EXPECTED_CHECKPOINT_SHA = handoff["teacher_source"]["checkpoint_sha256_locked"]

model_candidates = []
for manifest_path in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = read_json(manifest_path)
    except Exception:
        continue
    if payload.get("model_id") == "expansionnet_v2":
        model_candidates.append((manifest_path, payload))
assert len(model_candidates) == 1, model_candidates
MODEL_MANIFEST_PATH, model_manifest = model_candidates[0]
MODEL_ROOT = MODEL_MANIFEST_PATH.parent
CHECKPOINT_PATH = MODEL_ROOT / model_manifest["checkpoint"]["relative_path"]
NATIVE_VOCAB_PATH = MODEL_ROOT / model_manifest["vocabulary"]["relative_path"]
assert sha256(CHECKPOINT_PATH) == EXPECTED_CHECKPOINT_SHA
assert sha256(CHECKPOINT_PATH) == model_manifest["checkpoint"]["sha256"]

with NATIVE_VOCAB_PATH.open("rb") as handle:
    native_vocab = pickle.load(handle)
NATIVE_WORD2IDX = native_vocab["word2idx_dict"]
NATIVE_IDX2WORD = native_vocab["idx2word_list"]
NATIVE_BOS = int(NATIVE_WORD2IDX["SOS"])
NATIVE_EOS = int(NATIVE_WORD2IDX["EOS"])
NATIVE_PAD = int(NATIVE_WORD2IDX["PAD"])

SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
sys.path.insert(0, str(SOURCE_ROOT))
from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0)
teacher = End_ExpansionNet_v2(
    swin_img_size=384, swin_patch_size=4, swin_in_chans=3,
    swin_embed_dim=192, swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48], swin_window_size=12,
    swin_mlp_ratio=4.0, swin_qkv_bias=True, swin_qk_scale=None,
    swin_drop_rate=0.0, swin_attn_drop_rate=0.0, swin_drop_path_rate=0.0,
    swin_norm_layer=torch.nn.LayerNorm, swin_ape=False, swin_patch_norm=True,
    swin_use_checkpoint=False, final_swin_dim=1536, d_model=512,
    N_enc=3, N_dec=3, num_heads=8, ff=2048,
    num_exp_enc_list=[32, 64, 128, 256, 512], num_exp_dec=16,
    output_word2idx=NATIVE_WORD2IDX, output_idx2word=NATIVE_IDX2WORD,
    max_seq_len=74, drop_args=drop_args, rank=DEVICE,
)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
load_result = teacher.load_state_dict(checkpoint["model_state_dict"], strict=True)
assert not load_result.missing_keys and not load_result.unexpected_keys
del checkpoint
gc.collect()
teacher = teacher.to(DEVICE).eval()
for parameter in teacher.parameters():
    parameter.requires_grad_(False)

preprocess = transforms.Compose([
    transforms.Resize((384, 384), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
PILOT_IDS = train_ids[:128]

class PilotImages(Dataset):
    def __len__(self): return len(PILOT_IDS)
    def __getitem__(self, index):
        image_id = PILOT_IDS[index]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as source:
            image = preprocess(source.convert("RGB"))
        return image_id, image

def decode_native(ids):
    words = []
    for value in ids:
        value = int(value)
        if value == NATIVE_BOS: continue
        if value == NATIVE_EOS: break
        if value == NATIVE_PAD: continue
        words.append(str(NATIVE_IDX2WORD[value]))
    return " ".join(words).strip()

TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?|[^\w\s]", re.UNICODE)
TOKEN_TO_ID = vocab["token_to_id"]
def to_f30k_ids(caption):
    lexical = [x.lower() for x in TOKEN_RE.findall(caption.strip())][:29]
    return [1] + [int(TOKEN_TO_ID.get(x, 3)) for x in lexical] + [2]

loader = DataLoader(PilotImages(), batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
records = []
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
for image_ids, images in loader:
    images = images.to(DEVICE, non_blocking=True)
    with torch.inference_mode():
        predictions, _ = teacher(
            enc_x=images, enc_x_num_pads=[0] * images.shape[0],
            mode="beam_search", beam_size=3, beam_max_seq_len=31,
            sample_or_max="max", how_many_outputs=1,
            sos_idx=NATIVE_BOS, eos_idx=NATIVE_EOS, length_penalty=0.7,
        )
    for image_id, prediction in zip(image_ids.tolist(), predictions):
        native_ids = [int(x) for x in prediction[0]]
        caption = decode_native(native_ids)
        student_ids = to_f30k_ids(caption)
        assert caption and student_ids[0] == 1 and student_ids[-1] == 2
        assert len(student_ids) <= 31 and 0 not in student_ids
        records.append({
            "image_id": int(image_id), "caption": caption,
            "teacher_token_ids": native_ids, "student_token_ids": student_ids,
        })
elapsed = time.perf_counter() - started
assert len(records) == 128 and len({x["image_id"] for x in records}) == 128
rate = len(records) / elapsed
eta_hours = 29000 / rate / 3600
(WORK_ROOT / "pilot_captions.json").write_text(
    json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8"
)
RESULT = {
    "schema_version": "f30k_t1_pseudo_pilot_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed", "pilot_gate": "full_pseudocaption_generation_authorized",
    "teacher": "ExpansionNet v2", "matched_student": "S1 MobileNetV3-Large",
    "pilot_images": 128, "images_per_second": rate,
    "estimated_full_generation_hours": eta_hours,
    "checkpoint_sha256": sha256(CHECKPOINT_PATH),
    "beam_size": 3, "length_penalty": 0.7, "maximum_decoder_steps": 31,
    "flickr30k_token_ids_valid": True,
    "next_phase_authorized": "F30K_T1_Pseudo_Full",
    "test_set_status": "sealed", "test_set_used": False,
}
print("Pilot captions          : 128 / 128")
print("Images/second           :", round(rate, 4))
print("Estimated full runtime  :", round(eta_hours, 2), "hours")


/kaggle/input/models/nguynthnhdan/expansionnet-v2-ieee-bigdata2023/pytorch/default/1/expansionnet_v2_ieee_bigdata2023/ExpansionNet_v2_src/models/captioning_model.py:206: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  new_loop_dec_classes = loop_dec_classes[[bs_idxes, which_sequence]]
/kaggle/input/models/nguynthnhdan/expansionnet-v2-ieee-bigdata2023/pytorch/default/1/expansionnet_v2_ieee_bigdata2023/ExpansionNet_v2_src/models/captioning_model.py:207: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor

Pilot captions          : 128 / 128
Images/second           : 8.6516
Estimated full runtime  : 0.93 hours


## Cell 3 — Result và manifest SHA-256

In [3]:

RESULT_PATH = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(RESULT_PATH, RESULT)
artifacts = []
for path in sorted(p for p in WORK_ROOT.rglob("*") if p.is_file() and p.name != "MANIFEST.json"):
    artifacts.append({
        "relative_path": path.relative_to(WORK_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
MANIFEST_PATH = WORK_ROOT / "MANIFEST.json"
write_json(
    MANIFEST_PATH,
    {
        "schema_version": "f30k_teacher_workload_pilot_manifest_v1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_id": EXPERIMENT_ID,
        "status": RESULT["status"],
        "artifacts": artifacts,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
print("=" * 78)
print(EXPERIMENT_ID, "— FINAL")
print("=" * 78)
print("Status                  :", RESULT["status"])
print("Pilot gate              :", RESULT["pilot_gate"])
print("Next                    :", RESULT["next_phase_authorized"])
print("Held-out Test           : SEALED / NOT READ")
print("Output                  :", WORK_ROOT)


F30K_T1_Pseudo_Pilot — FINAL
Status                  : passed
Pilot gate              : full_pseudocaption_generation_authorized
Next                    : F30K_T1_Pseudo_Full
Held-out Test           : SEALED / NOT READ
Output                  : /kaggle/working/F30K_T1_Pseudo_Pilot
